# Análise Facial de Vídeos do YouTube

Este notebook realiza análise demográfica dos criadores de conteúdo através da extração e análise facial de frames de vídeos.

**Pipeline**: Download de vídeos → Extração de frames → Detecção facial (InsightFace) → Extração de embeddings (MagFace) → Clustering (DBSCAN) → Análise demográfica

**Resultado**: CSV com idade, gênero, consistência e métricas de qualidade para cada vídeo

In [1]:
import pandas as pd
import numpy as np
import requests
import os
from tqdm.notebook import tqdm
import subprocess
import time
import scenedetect
import cv2
import torch
import sys
from torch.nn.modules.utils import consume_prefix_in_state_dict_if_present
from insightface.app import FaceAnalysis
from insightface.utils import face_align
from sklearn.cluster import DBSCAN
import matplotlib.pyplot as plt
from collections import Counter
from statistics import mode, median

sys.path.append('./videos_files')
from iresnet import iresnet50

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [2]:
project_base_path = './videos_files'
log_videos_path = os.path.join(project_base_path, 'videos_log.txt')

required_folders_relative = [
    'videos'
]

for folder_name in required_folders_relative:
    full_folder_path = os.path.join(project_base_path, folder_name)
    if not os.path.exists(full_folder_path):
        os.makedirs(full_folder_path)
        print(f"Pasta criada: {full_folder_path}")

In [3]:
df = pd.read_csv('resumos/df_resumos_complete_cleaned.csv')

In [4]:
caminho_pesos = './videos_files/magface_iresnet50_MS1MV2_ddp_fp32.pth'
checkpoint = torch.load(caminho_pesos, map_location=device)

model = iresnet50()

if 'state_dict' in checkpoint:
    state_dict = checkpoint['state_dict']
else:
    state_dict = checkpoint

consume_prefix_in_state_dict_if_present(state_dict, prefix="features.module.")

model.load_state_dict(state_dict, strict=False)

model.to(device)
model.eval()

IResNet(
  (conv1): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
  (bn1): BatchNorm2d(64, eps=2e-05, momentum=0.9, affine=True, track_running_stats=True)
  (prelu): PReLU(num_parameters=64)
  (layer1): Sequential(
    (0): IBasicBlock(
      (bn1): BatchNorm2d(64, eps=2e-05, momentum=0.9, affine=True, track_running_stats=True)
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=2e-05, momentum=0.9, affine=True, track_running_stats=True)
      (prelu): PReLU(num_parameters=64)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (bn3): BatchNorm2d(64, eps=2e-05, momentum=0.9, affine=True, track_running_stats=True)
      (downsample): Sequential(
        (0): Conv2d(64, 64, kernel_size=(1, 1), stride=(2, 2), bias=False)
        (1): BatchNorm2d(64, eps=2e-05, momentum=0.9, affine=True, track_running_stats=True)
      )
    )
    (1): IBas

---
## 1. Download de Vídeos

Baixa vídeos do YouTube usando **yt-dlp**. Sistema de checkpoint evita redownload de vídeos já processados.

In [5]:
def save_progress(video_id, output_file):
    with open(output_file, 'a') as f:
        f.write(f"{video_id}\n")
        
def load_progress(output_file):
    if os.path.exists(output_file):
        with open(output_file, 'r') as f:
            return set(line.strip() for line in f)
    return set()

In [6]:
completed_videos = load_progress(log_videos_path)

video_folder = os.path.join(project_base_path, 'videos')

In [ ]:
for video_id in tqdm(df['video_id'].drop_duplicates(), desc="Baixando vídeos", unit="vídeo"):
    output_file = os.path.join(video_folder, f"{video_id}.mp4")

    if os.path.exists(output_file) or video_id in completed_videos:
        print(f"Vídeo {video_id} já baixado ou registrado.")
        continue

    video_url = f"https://www.youtube.com/watch?v={video_id}"

    try:
        print(f"Baixando vídeo {video_id}...")
        command = [
            'yt-dlp',
            '-f', 'bestvideo[vcodec^=avc1][ext=mp4]',
            '-o', output_file,
            '--no-warnings',
            '--retries', '5',
            video_url
        ]
        
        result = subprocess.run(command, capture_output=True, text=True, check=True)
        
        save_progress(video_id, log_videos_path)
        print(f"Vídeo {video_id} baixado com sucesso.")
        
    except subprocess.CalledProcessError as e:
        print(f"Erro ao baixar o vídeo {video_id}: {e.stderr.strip()}")
        continue

print("Todos os vídeos foram processados.")

---
## 2. Extração e Análise de Frames

Esta seção implementa o pipeline completo de análise facial:

### Etapas:
1. **Detecção Facial** (`extract_face_frames`): Extrai 1 frame por segundo e detecta rostos usando InsightFace (buffalo_l), retornando faces alinhadas (112x112), idade e gênero estimados
2. **Extração de Embeddings** (`get_face_features`): Processa cada rosto com modelo MagFace (iresnet50) para gerar embeddings normalizados de 512 dimensões, incluindo magnitude como métrica de qualidade
3. **Clustering** (`faces_clustering`): Agrupa rostos similares usando DBSCAN (métrica cosine) para identificar o apresentador principal vs. aparições secundárias
4. **Análise Demográfica** (`get_final_demographics`): Seleciona os top-5 frames de maior qualidade do cluster principal e calcula:
   - **Idade**: Mediana das idades detectadas
   - **Gênero**: Gênero mais frequente
   - **Consistência de Gênero**: Proporção do gênero dominante (1.0 = único gênero)
   - **Cluster Percentage**: % de frames que o apresentador principal representa
   - **Quality Score**: Magnitude média dos embeddings (confiança da detecção)

### Métricas de Output:
- **age_std**: Desvio padrão das idades (consistência etária)
- **gender_consistency**: Valores < 1 indicam inconsistência
- **cluster_percentage**: Alta % indica vídeo com um único apresentador

In [7]:
app = FaceAnalysis(name="buffalo_l", providers=['CUDAExecutionProvider', 'CPUExecutionProvider'])
app.prepare(ctx_id=0, det_size=(640, 640))

def extract_face_frames(video_path: str, frames_per_second: int = 1, min_confidence: float = 0.80):
    """
    Extrai frames com rostos detectados de um vídeo.
    
    Args:
        video_path: Caminho completo do arquivo de vídeo (.mp4)
        frames_per_second: Quantos frames processar por segundo (default: 1)
        min_confidence: Confiança mínima da detecção facial (0-1, default: 0.80)
    
    Returns:
        Lista de dicionários, cada um contendo:
            - aligned_face: Imagem do rosto alinhado (112x112 pixels, NumPy array BGR)
            - age: Idade estimada pelo InsightFace (int)
            - gender: Gênero estimado ('M' ou 'F')
    """
    cap = cv2.VideoCapture(video_path)
    fps_original = cap.get(cv2.CAP_PROP_FPS)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    
    if fps_original == 0:
        print(f"Erro: não foi possível obter FPS do vídeo {video_path}")
        cap.release()
        return []
    
    # Calcula o intervalo entre frames a processar
    frame_interval = int(fps_original / frames_per_second)
    if frame_interval < 1:
        frame_interval = 1
    
    face_data = []
    frame_count = 0

    while frame_count < total_frames:
        cap.set(cv2.CAP_PROP_POS_FRAMES, frame_count)
        sucesso, frame = cap.read()
        
        if not sucesso:
            break

        # Detecta faces no frame
        faces = app.get(frame)
        
        # Para cada face detectada, extrai e alinha
        for face in faces:
            # Filtra faces com baixa confiança
            if face.det_score < min_confidence:
                continue
            
            aligned_face = face_align.norm_crop(frame, landmark=face.kps)
            
            face_data.append({
                'aligned_face': aligned_face,
                'age': face.age,
                'gender': "M" if face.gender == 1 else "F"
            })
            
        frame_count += int(fps_original / frames_per_second)

    cap.release()
    return face_data

/home/gabrielct/miniconda3/envs/myenv/lib/python3.11/site-packages/onnxruntime/capi/onnxruntime_inference_collection.py:123: UserWarning: Specified provider 'CUDAExecutionProvider' is not in available provider names.Available providers: 'AzureExecutionProvider, CPUExecutionProvider'
  warnings.warn(


Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /home/gabrielct/.insightface/models/buffalo_l/1k3d68.onnx landmark_3d_68 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /home/gabrielct/.insightface/models/buffalo_l/2d106det.onnx landmark_2d_106 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /home/gabrielct/.insightface/models/buffalo_l/det_10g.onnx detection [1, 3, '?', '?'] 127.5 128.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /home/gabrielct/.insightface/models/buffalo_l/genderage.onnx genderage ['None', 3, 96, 96] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /home/gabrielct/.insightface/models/buffalo_l/w600k_r50.onnx recognition ['None', 3, 112, 112]

In [8]:
def preprocess_face_for_model(face_img):
    """
    Pré-processa imagem facial para o modelo iresnet50.
    
    Args:
        face_img: Imagem do rosto (NumPy array BGR)
    
    Returns:
        Tensor PyTorch [1, 3, 112, 112] normalizado em [-1, 1]
    """
    # Converte BGR para RGB
    face_rgb = cv2.cvtColor(face_img, cv2.COLOR_BGR2RGB)
    
    # Normaliza para [-1, 1]
    face_normalized = (face_rgb.astype(np.float32) / 127.5) - 1.0
    
    # Transpõe de (H, W, C) para (C, H, W)
    face_transposed = np.transpose(face_normalized, (2, 0, 1))
    
    # Converte para tensor e adiciona batch dimension
    face_tensor = torch.from_numpy(face_transposed).unsqueeze(0).to(device)
    
    return face_tensor


def get_face_features(faces: list):
    """
    Extrai features (embeddings) de uma lista de rostos.
    
    Args:
        faces: Lista de dicionários com 'aligned_face', 'age', 'gender'
    
    Returns:
        Lista de dicionários contendo:
            - embedding_norm: embedding normalizado (L2)
            - magnitude: magnitude do embedding original
            - face_img: imagem original do rosto
            - age: idade detectada
            - gender: gênero detectado
    """
    results = []
    
    for face_data in faces:
        face_img = face_data['aligned_face']
        face_tensor = preprocess_face_for_model(face_img)
        
        with torch.no_grad():
            embedding_tensor = model(face_tensor)
            
            # Calcula magnitude do embedding (útil para qualidade)
            magnitude = torch.norm(embedding_tensor, p=2).item()
            
            # Converte para numpy
            embedding_raw = embedding_tensor.cpu().numpy().flatten()
            
            # Normalização L2 para comparação por similaridade coseno
            embedding_norm = embedding_raw / np.linalg.norm(embedding_raw)
            
            results.append({
                'embedding_norm': embedding_norm,
                'magnitude': magnitude,
                'face_img': face_img,
                'age': face_data['age'],
                'gender': face_data['gender']
            })
            
    return results


def faces_clustering(list_embeddings_norm: list, eps: float = 0.4, min_samples: int = 2):
    """
    Agrupa embeddings faciais usando DBSCAN.
    
    Args:
        list_embeddings_norm: Lista de arrays de embeddings normalizados
        eps: Distância máxima entre amostras
        min_samples: Número mínimo de amostras para formar um cluster
    
    Returns:
        Array de labels (-1 para outliers, >=0 para IDs de clusters)
    """
    if len(list_embeddings_norm) == 0:
        return np.array([])
    
    dbscan = DBSCAN(eps=eps, min_samples=min_samples, metric='cosine')
    labels = dbscan.fit_predict(list_embeddings_norm)
    
    return labels

def get_final_demographics(features: list, labels: list, top_n: int = 5):
    """
    Usa a magnitude do MagFace para filtrar os melhores frames e decidir idade/gênero.
    
    Args:
        features: Lista contendo as informações de cada frame processado
        labels: Lista de identificadores de cada cluster encontrado
        top_n: Número máximo de melhores frames a serem selecionados
    """
    if len(labels) == 0: return None
    
    # Identificar o maior cluster (Apresentador)
    valid_labels = [l for l in labels if l != -1]
    if not valid_labels: return None
    
    largest_cluster_id = Counter(valid_labels).most_common(1)[0][0]
    
    # Filtrar rostos desse cluster e ordenar pela qualidade (magnitude)
    cluster_faces = [f for f, l in zip(features, labels) if l == largest_cluster_id]
    cluster_faces.sort(key=lambda x: x['magnitude'], reverse=True)
    
    # Pegar os Top N melhores frames
    top_n = min(len(cluster_faces), top_n)
    best_samples = cluster_faces[:top_n]
    
    # Calcular a porcentagem que o maior cluster representa
    total_faces = len(labels)
    largest_cluster_size = len(cluster_faces)
    cluster_percentage = (largest_cluster_size / total_faces) * 100
    
    genders = [s['gender'] for s in best_samples]
    gender_counts = Counter(genders)
    ages = [s['age'] for s in best_samples]
    avg_magnitude = sum(s['magnitude'] for s in best_samples) / top_n
    
    results = {
        'gender': gender_counts.most_common(1)[0][0],
        'age': int(median(ages)),
        'age_std': round(np.std(ages), 2),
        'gender_consistency': round(gender_counts.most_common(1)[0][1] / len(genders), 2),
        'quality_score': round(avg_magnitude, 2),
        'cluster_percentage': round(cluster_percentage, 2)
    }
    
    return results
    
def save_info_csv(video_id: str, csv_path: str, results: dict):
    """
    Salva as informações finais em um CSV
    
    Args:
        video_id: Identificador único de cada vídeo
        csv_path: Local onde o arquivo será salvo
        results: Informações a serem salvas dentro do CSV
    """    
    gender = results['gender']
    age = results['age']
    age_std = results['age_std']
    gender_consistency = results['gender_consistency']
    quality = results['quality_score']
    cluster_percentage = results['cluster_percentage']
    
    if not os.path.exists(csv_path):
        with open(csv_path, 'w') as f:
            f.write('video_id,age,gender,age_std,gender_consistency,quality_magnitude,cluster_percentage\n')

    with open(csv_path, 'a') as f:
        f.write(f"{video_id},{age},{gender},{age_std},{gender_consistency},{quality},{cluster_percentage}\n")

In [ ]:
csv_path = os.path.join(project_base_path, 'videos_demographics_complete.csv')

# Carregar vídeos já processados
processed_videos = set()
if os.path.exists(csv_path):
    df_processed = pd.read_csv(csv_path)
    processed_videos = set(df_processed['video_id'].values)

video_files = [f for f in os.listdir(video_folder) if f.endswith('.mp4')]

for video_filename in tqdm(video_files, desc='Analisando frames...'):
    video_id = os.path.splitext(video_filename)[0]
    
    if video_id in processed_videos:
        continue
    
    video_path = os.path.join(video_folder, video_filename)
    
    try:
        # Extrair faces
        faces = extract_face_frames(video_path)
        
        if len(faces) == 0:
            print(f"{video_id}: Nenhuma face detectada")
            continue
        
        # Extrair features
        faces_features = get_face_features(faces)
        
        # Clustering
        embeddings_list = [f['embedding_norm'] for f in faces_features]
        clusters = faces_clustering(embeddings_list)
        
        # Obter demografia final
        demographics = get_final_demographics(faces_features, clusters)
        
        if demographics is None:
            print(f"{video_id}: Não foi possível determinar demografia")
            continue
        
        # Salvar no CSV
        save_info_csv(video_id, csv_path, demographics)
        
    except Exception as e:
        print(f"Erro ao processar {video_id}: {str(e)}")
        continue